## Set up the input datasets

In [ ]:
import pandas as pd

In [ ]:
# Load the CSV file with labels
labels_df = pd.read_csv("/kaggle/input/bhf-data-science-centre-ecg-challenge/train_final.csv")

# Some images are broken/unreadable
# Use these lines to only read in the "valid" images to stop 
with open('/kaggle/input/bhf-reference-files/valid_images_list.txt', 'r') as file:
    valid_image_paths = [line.strip() for line in file]

with open('/kaggle/input/bhf-reference-files/valid_test_images_list.txt', 'r') as file:
    valid_test_images = [line.strip() for line in file]

## Build a custom dataset class and dataloaders

In [ ]:
import sys 
sys.path.append("/kaggle/input/seg-model-module/")
from seg_model import get_model, deep_learning_scan

import os

import torch
import torchvision.transforms as T
from torchvision.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt 

In [ ]:
# work out device 
device = torch.device("mps" if torch.backends.mps.is_available() \
                      else ("cuda" if torch.cuda.is_available() 
                            else "cpu"))

In [ ]:
seg_model_path = '/kaggle/input/document-detection/pytorch/default/1/model_mbv3_iou_mix_2C049.pth'
seg_model = get_model(model_path, device) # get image segmentation model 
seg_model.eval() # put model in evaluation mode

In [ ]:
class ECG_Dataset(Dataset):
    def __init__(self, image_paths, label_df, transforms=None, preprocess_fn=None, seg_model=None):
        self.image_paths = image_paths # list of valid image paths 
        self.labels_df = label_df # dataframe containing the labels that we are going to use
        self.transforms = transforms # which transforms to apply to the data
        self.preprocess_fn = preprocess_fn # function that essentially calls the segmentation model
        self.seg_model = seg_model
        
    def __len__(self,):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Failed to load image: {image_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if ((self.preprocess_fn is not None) and (self.seg_model is not None)): 
            image = self.preprocess_fn(image, self.seg_model) # uses seg model to extract the ECG image

        if (self.transforms is not None): 
            # apply transformations to image 
            image = self.transforms(image)
            
        # labels object is a dataframe so need to convert to a list format for classification 
        # get the image ID label (through some tedious image manipulation)
        filename = os.path.basename(image_path)  # extracts 'train_XXXXXX.png' from full path
        index = int(filename.rsplit('_', 1)[-1].split('.')[0])  # extracts 'XXXXXX' and converts to int

        row = self.labels_df[self.labels_df['ID'] == index].iloc[:, 1:].values  # return row with the idx value
        label = torch.tensor(row, dtype=torch.float32)  # Convert to PyTorch tensor for training 

        return image, label

In [ ]:
train_transform = T.Compose([
                        T.resize((512, 512)),
                        T.ToTensor(), 
                        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                        ])

val_transform = T.Compose([
                        T.resize((512, 512)),
                        T.ToTensor(), 
                        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                        ])

In [ ]:
# Create the datasets for training and validation
train_dataset = ECG_Dataset(valid_image_paths, labels_df, transform=train_transform)
val_dataset = ECG_Dataset(valid_test_images, labels_df, transform=val_transform)

In [ ]:
# Create the dataloaders 
train_dataloader = DataLoader(train_dataset, 
                             num_workers = 4,
                             pin_memory = True, 
                             shuffle = True, 
                             batch_size = 1)

val_dataloader = DataLoader(val_dataset, 
                             num_workers = 4,
                             pin_memory = True, 
                             shuffle = True, 
                             batch_size = 1)

In [ ]:
# Visualise an image from the dataloader and see whether it is sensible 

dataiter = iter(train_dataloader)  # Get an iterator from the dataloader
image, label = next(dataiter)  # Get one batch

# Convert the tensor to a NumPy array for visualization
image = image[0].permute(1, 2, 0).cpu().numpy()  # Convert (C, H, W) → (H, W, C)

# Unnormalize if needed (assuming mean=[0.5, 0.5, 0.5] and std=[0.5, 0.5, 0.5])
image = image * 0.5 + 0.5  # Convert from [-1,1] back to [0,1]

# Display the image
plt.imshow(image)
plt.axis("off")
plt.title(f"Label: {label[0].tolist()}")
plt.show()
